# Diagnosing AttributeError: name in nomad-cau-plugin Tests

This notebook analyzes and resolves the test failures that occur after updating nomad-cau-plugin in Docker.

## Summary

- **Error**: `AttributeError: name` in NOMAD's metainfo system when parsing test archive files
- **Failing Tests**: `test_schema_package` and `test_michaela_schema_package`
- **Symptom**: NOMAD's `_resolve_name()` function tries to access `.name` attribute on an `EntryArchive(metadata)` object that doesn't have it
- **Root Cause**: Schema proxy resolution fails; likely due to incomplete section class registration or schema incompatibility

## Section 1: Parse Error Analysis

Analyze the traceback from the failed test to identify where the failure occurs in the parsing stage.

In [ ]:
# Error Traceback Analysis
error_trace = """
venv/lib/python3.12/site-packages/nomad/metainfo/metainfo.py:333: in _resolve_name
    if context.name == name and context != self.m_proxy_section:
       ^^^^^^^^^^^^
    
FAILED: AttributeError: name
self = EntryArchive(metadata), name = 'name'
"""

print("KEY FINDINGS FROM ERROR TRACE:")
print("=" * 60)
print("1. Location: _resolve_name() in NOMAD's metainfo/metainfo.py:333")
print("2. Context: EntryArchive(metadata) object")
print("3. Problem: Trying to access '.name' attribute")
print("4. Result: AttributeError because EntryArchive(metadata) has no 'name'")
print("\n" + "=" * 60)
print("\nINTERPRETATION:")
print("The metainfo proxy resolution system is trying to resolve section")
print("references but encounters a section proxy that:")
print("  - Is an EntryArchive metadata object")
print("  - Lacks a 'name' attribute")
print("  - Causes the recursive _resolve_name() to fail")
print("\nThis typically indicates:")
print("  ✗ Schema section definitions are incomplete")
print("  ✗ Section class registration failed")
print("  ✗ Forward references couldn't be resolved")
print("  ✗ NOMAD version incompatibility with schema")


## Section 2: Schema Definition Validation

Verify that schema definitions are properly configured and compatible with NOMAD.

In [ ]:
import sys
import os

# Add plugin path
plugin_path = '/home/lankovas/nomad-distro-dev/packages/nomad-cau-plugin/src'
sys.path.insert(0, plugin_path)

print("Checking schema package structure:")
print("=" * 60)

try:
    from nomad_cau_plugin.schema_packages.schema_package import (
        NewSchemaPackage,
        Michaela,
        Characterization,
        IRMeasurement,
        DLSMeasurement,
    )
    
    print("\n✓ Successfully imported all schema classes:")
    print(f"  - NewSchemaPackage: {NewSchemaPackage}")
    print(f"  - Michaela: {Michaela}")
    print(f"  - Characterization: {Characterization}")
    print(f"  - IRMeasurement: {IRMeasurement}")
    print(f"  - DLSMeasurement: {DLSMeasurement}")
    
    # Check for required attributes
    print("\n" + "=" * 60)
    print("Checking for 'name' field in schema classes:")
    for cls in [NewSchemaPackage, Michaela, Characterization]:
        has_name = hasattr(cls, 'name')
        print(f"  {cls.__name__}: {'✓ has name field' if has_name else '✗ MISSING name field'}")
    
except ImportError as e:
    print(f"✗ Failed to import schema classes: {e}")
    import traceback
    traceback.print_exc()


## Section 3: Archive YAML Structure Inspection

Load and inspect the test archive files to verify they conform to expected schema.

In [ ]:
import yaml

test_dir = '/home/lankovas/nomad-distro-dev/packages/nomad-cau-plugin/tests/data'

# Load test.archive.yaml
test_file = os.path.join(test_dir, 'test.archive.yaml')
print("=" * 60)
print("1. Analyzing: test.archive.yaml")
print("=" * 60)

if os.path.exists(test_file):
    with open(test_file, 'r') as f:
        test_data = yaml.safe_load(f)
    print("\nYAML Structure:")
    print(yaml.dump(test_data, default_flow_style=False))
    print(f"\nm_def: {test_data.get('data', {}).get('m_def')}")
    print(f"Fields: {list(test_data.get('data', {}).keys())}")
else:
    print(f"✗ File not found: {test_file}")

print("\n" + "=" * 60)
print("2. Analyzing: test_michaela.archive.yaml")
print("=" * 60)

test_michaela_file = os.path.join(test_dir, 'test_michaela.archive.yaml')
if os.path.exists(test_michaela_file):
    with open(test_michaela_file, 'r') as f:
        test_michaela_data = yaml.safe_load(f)
    print("\nYAML Structure (first 30 lines):")
    yaml_str = yaml.dump(test_michaela_data, default_flow_style=False)
    print('\n'.join(yaml_str.split('\n')[:30]))
    print("\n...")
    print(f"\nm_def: {test_michaela_data.get('data', {}).get('m_def')}")
    print(f"Top-level fields: {list(test_michaela_data.get('data', {}).keys())}")
else:
    print(f"✗ File not found: {test_michaela_file}")


## Section 4: Metainfo Proxy Resolution Debugging

Examine how NOMAD's metainfo system resolves section class definitions.

In [ ]:
print("Checking m_package registration:")
print("=" * 60)

try:
    from nomad_cau_plugin.schema_packages.schema_package import m_package
    
    print(f"\n✓ m_package loaded: {m_package}")
    print(f"  Package name: {m_package.name if hasattr(m_package, 'name') else 'N/A'}")
    
    # Check registered sections
    if hasattr(m_package, 'packages'):
        print(f"\n  Sub-packages: {len(m_package.packages)}")
        for pkg in m_package.packages:
            print(f"    - {pkg}")
    
    if hasattr(m_package, 'sections'):
        print(f"\n  Registered sections: {len(m_package.sections)}")
        for section in m_package.sections:
            print(f"    - {section.name if hasattr(section, 'name') else 'UNNAMED'}")
    else:
        print("\n  WARNING: No sections attribute found!")
    
    # Check metainfo initialization
    print(f"\n  Metainfo initialized: {hasattr(m_package, '_initialized')}")
    
except Exception as e:
    print(f"✗ Error loading m_package: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("Checking SubSection references in Characterization:")
print("=" * 60)

try:
    from nomad_cau_plugin.schema_packages.schema_package import Characterization
    
    if hasattr(Characterization, 'm_def'):
        m_def = Characterization.m_def
        print(f"\nCharacterization.m_def: {m_def}")
        
        # Check all attributes
        print("\nAll attributes:")
        for attr_name in dir(Characterization):
            if not attr_name.startswith('_'):
                attr = getattr(Characterization, attr_name)
                if hasattr(attr, 'section_def'):
                    print(f"  - {attr_name}: SubSection of {attr.section_def}")
    else:
        print("WARNING: No m_def found")
        
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()


## Section 5: Root Cause Diagnosis & Solution

Based on the analysis above, identify the root cause and implement the fix.

### Docker / CI Failure Log

The GitHub Actions run updates `nomad-cau-plugin` to commit `45f19af` and then fails while parsing `tests/schema_packages/test_schema_package.py` and `tests/schema_packages/test_schema_package.py::test_michaela_schema_package`.

Observed pattern:
- Environment setup succeeds
- Plugin installs successfully
- Parser and IR/DLS tests pass
- Schema package tests fail during archive parsing with `AttributeError: name`
- Failure occurs inside NOMAD metainfo proxy resolution, not in the plugin's own normalizer code

## Diagnosis and Next Steps

Most likely root cause:

- The plugin installs successfully, so this is not a packaging or dependency download failure.
- The failure happens when NOMAD parses archive YAML and tries to resolve a subsection class during metainfo proxy resolution.
- `AttributeError: name` on `EntryArchive(metadata)` strongly suggests a broken or incompatible metainfo proxy chain, usually from one of these:
  - a section class not registered in the current `SchemaPackage`
  - a circular import that changes import order in the Docker image
  - a NOMAD version change that makes the schema definition incompatible

Recommended checks:

1. Rebuild the Docker image without cache to rule out stale plugin metadata.
2. Confirm the exact NOMAD version in the image matches the version used for local tests.
3. Verify that `schema_package.py` loads all section classes before `m_package.__init_metainfo__()` runs.
4. Confirm the fully-qualified `m_def` path in the archive YAML still matches the current module path.
5. If the error started after moving schema classes across files, check for circular imports between `schema_packages`, `measurements`, and `normalizers`.